Importing ORM utilities and declaring entities:

In [11]:
from enum import Enum
from sqlalchemy import ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
  pass

class Project(Base):
  __tablename__ = 'Project'
  project_id: Mapped[int] = mapped_column(primary_key=True)
  project_name: Mapped[str] = mapped_column()
  enhancement_proposal_name: Mapped[str] = mapped_column()
  copyright: Mapped[str] = mapped_column()

class Person(Base):
  __tablename__ = 'Person'
  person_id: Mapped[int] = mapped_column(primary_key=True)
  full_name: Mapped[str] = mapped_column()

class PersonIdentifier(Base):
  __tablename__ = 'PersonIdentifier'
  person_id: Mapped[int] = mapped_column(ForeignKey('Person.person_id'), primary_key=True)
  domain: Mapped[str] = mapped_column(primary_key=True)
  identifier_type: Mapped[str] = mapped_column(primary_key=True)
  identifier: Mapped[str] = mapped_column(primary_key=True)

class Proposal(Base):
  __tablename__ = 'Proposal'
  project_id: Mapped[int] = mapped_column(ForeignKey('Project.project_id'), primary_key=True)
  proposal_id: Mapped[str] = mapped_column(primary_key=True)
  topic: Mapped[str] = mapped_column()
  proposal_type: Mapped[str] = mapped_column()

class NormalisedStatus(str, Enum):
  accepted = 'accepted'
  rejected = 'rejected'
  draft = 'draft'
  review = 'review'
  withdrawn = 'withdrawn'
  superseded = 'superseded'
  unknown = 'unknown'

class ProposalStatus(Base):
  __tablename__ = 'ProposalStatus'
  project_id: Mapped[int] = mapped_column(ForeignKey('Proposal.project_id'), primary_key=True)
  proposal_id: Mapped[str] = mapped_column(ForeignKey('Proposal.proposal_id'), primary_key=True)
  status_index: Mapped[int] = mapped_column(primary_key=True)
  normalised_status: Mapped[NormalisedStatus] = mapped_column()
  raw_status: Mapped[str] = mapped_column()
  created_at: Mapped[str] = mapped_column(nullable=False)

class ProposalRevision(Base):
  __tablename__ = 'ProposalRevision'
  project_id: Mapped[int] = mapped_column(ForeignKey('Proposal.project_id'), primary_key=True)
  proposal_id: Mapped[str] = mapped_column(ForeignKey('Proposal.proposal_id'), primary_key=True)
  revision_index: Mapped[int] = mapped_column(primary_key=True)
  title: Mapped[str] = mapped_column(nullable=False)
  created_at: Mapped[str] = mapped_column(nullable=False)
  content: Mapped[str] = mapped_column()
  implemented_at_version: Mapped[str] = mapped_column()

class RelationType(str, Enum):
  related = 'related'
  supersedes = 'supersedes'

class RelatedProposal(Base):
  __tablename__ = 'RelatedProposal'
  project_id: Mapped[int] = mapped_column(ForeignKey('Proposal.project_id'), primary_key=True)
  proposal_id: Mapped[str] = mapped_column(ForeignKey('Proposal.proposal_id'), primary_key=True)
  related_project_id: Mapped[int] = mapped_column(ForeignKey('Proposal.project_id'), primary_key=True)
  related_proposal_id: Mapped[str] = mapped_column(ForeignKey('Proposal.proposal_id'), primary_key=True)
  relation_type: Mapped[RelationType] = mapped_column(name='type')

class Comment(Base):
  __tablename__ = 'Comment'
  comment_id: Mapped[int] = mapped_column(primary_key=True)
  author_id: Mapped[int] = mapped_column(nullable=False)
  project_id: Mapped[int] = mapped_column(ForeignKey('Proposal.project_id'), nullable=False)
  proposal_id: Mapped[str] = mapped_column(ForeignKey('Proposal.proposal_id'), nullable=False)
  comment_on_comment_id: Mapped[int] = mapped_column(ForeignKey('Comment.comment_id'))
  created_at: Mapped[str] = mapped_column()
  content: Mapped[str] = mapped_column(nullable=False)


Write the path to your database file here:

In [12]:
db_file = 'data/stuff.db'

In [13]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

eng = create_engine(f'sqlite:///{db_file}')
Base.metadata.create_all(eng)
session = Session(eng)

In [14]:
from matplotlib import pyplot as plt

In [15]:
from sqlalchemy import and_, select

for project in session.scalars(select(Project)):
  prop_ids = list(session.scalars(select(Proposal.proposal_id)
    .where(Proposal.project_id.is_(project.project_id))))
  prop_accepted = [prop_id for prop_id in prop_ids
    if session.scalar(select(ProposalStatus)
      .where(and_(
        ProposalStatus.project_id.is_(project.project_id),
        ProposalStatus.proposal_id.is_(prop_id),
        ProposalStatus.normalised_status.is_('accepted'))))]
  # print(prop_accepted)
  # print(prop_ids)